# 🎬 Project: YouTube Summarization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nawooooojin/openai-api-tutorial-kr/blob/main/3_Youtube_Summarization_Project/youtube_summarization_project.ipynb)

- **프로젝트 목적**: 유튜브 비디오에서 추출한 오디오를 입력으로 받아 전사 → 요약 → 이미지 생성 → PDF 요약본 생성까지의 전체 흐름을 구현

## 1. 환경 준비 및 API 키 설정

In [ ]:
%pip install -q -U openai yt-dlp pydantic reportlab pillow

In [ ]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata  # type: ignore
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass  # Colab이 아닌 환경
    except Exception:
        print("Colab 메뉴의 보안 비밀(열쇠 아이콘)에 OPENAI_API_KEY가 저장되어 있는지, 노트북 액세스가 켜져 있는지 확인해주세요.")

if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("OPENAI_API_KEY를 입력하세요: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API 키 설정 완료")

In [ ]:
import os

base_dirs = [
    ("fonts", []),         
    ("data", ["audio"]),       
    ("results", ["images", "audio", "transcribe"]),
]

for root, subfolders in base_dirs:
    os.makedirs(root, exist_ok=True)
    print(f"ready: {root}")

    for sub in subfolders:
        path = os.path.join(root, sub)
        os.makedirs(path, exist_ok=True)
        print(f"ready: {path}")

In [ ]:
import urllib.request
from pathlib import Path

from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

FONT_BASE_URL = "https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/"
FONT_FILES = ["NanumGothic-Regular.ttf", "NanumGothic-Bold.ttf", "NanumGothic-ExtraBold.ttf"]


def download_korean_fonts(font_dir="fonts"):
    """PDF에 한글을 쓰기 위해 나눔고딕 폰트(SIL OFL)를 내려받습니다. 이미 있으면 건너뜁니다."""
    font_dir = Path(font_dir)
    font_dir.mkdir(exist_ok=True)
    for font_file in FONT_FILES:
        font_path = font_dir / font_file
        if not font_path.exists():
            urllib.request.urlretrieve(FONT_BASE_URL + font_file, font_path)
            print(f"⬇️ {font_file} 다운로드 완료")


def register_korean_fonts(font_dir="fonts"):
    """나눔고딕 폰트를 등록합니다.

    Returns:
        dict: {'regular': 'KoreanFontRegular', 'bold': 'KoreanFontBold', 'extrabold': 'KoreanFontExtraBold'}
    """
    download_korean_fonts(font_dir)

    font_configs = {
        "regular": ("KoreanFontRegular", "NanumGothic-Regular.ttf"),
        "bold": ("KoreanFontBold", "NanumGothic-Bold.ttf"),
        "extrabold": ("KoreanFontExtraBold", "NanumGothic-ExtraBold.ttf"),
    }

    registered_fonts = {}
    for key, (font_name, font_file) in font_configs.items():
        font_path = Path(font_dir) / font_file
        pdfmetrics.registerFont(TTFont(font_name, str(font_path)))
        registered_fonts[key] = font_name

    print("✅ 모든 나눔고딕 폰트 등록 완료")
    return registered_fonts

In [ ]:
from pathlib import Path
from openai import OpenAI

client = OpenAI()
print("✅ 클라이언트 생성 완료")

## 2. YouTube 오디오 추출

- **목적**
  - 유튜브 강의 링크에서 **오디오 스트림만 추출**하여 로컬 파일로 저장
- **결과 예시**
  - `data/audio/3b1b_llm_lecture.webm` 형태로 저장


In [ ]:
url = "https://youtu.be/LPZh9BOjkQs" # 유튜브 링크

!yt-dlp -f "bestaudio[ext=webm]" -o "data/audio/3b1b_llm_lecture.webm" "$url"

## 3. 요약용 프롬프트 정의

- **역할**
  - 유튜브 전사 텍스트를 LLM에 입력할 때 사용할 시스템 / 유저 프롬프트 템플릿을 정의
  - LLM이 일관된 JSON 구조로 응답을 생성하도록 가이드

- **구성**
  - `SUMMARY_SYSTEM_PROMPT`: LLM의 역할과 출력 형식을 지정하는 시스템 프롬프트
  - `SUMMARY_USER_PROMPT_TEMPLATE`: 전사본(`{transcript_snippet}`)과 JSON 형식 가이드를 포함하는 유저 프롬프트 템플릿

- **포인트**
  - 강의 내용을 몇 개의 주요 토픽(topic)으로 나누도록 지시
  - 각 토픽마다 `title`, `summary`, `image_prompt` 3개 필드를 포함하도록 지시
  - 최종 출력(요약 및 토픽 설명)은 모두 한국어로 작성되도록 지시

In [ ]:
# ----------------------------------------------------
# TODO 1: SUMMARY_SYSTEM_PROMPT 완성
# ----------------------------------------------------
# 요구사항:
# - 전체 요약 지시사항 포함해야합니다.
# - topic 분할 요청 포함해야합니다. (토픽 분할 개수는 자유, 예시 4~6개)
# - topic별 title, summary, image_prompt 작성하도록 지시해야합니다.
# - JSON 구조 출력 명시해야합니다.
# - 모든 출력은 한국어로 작성해달라고 지시해야합니다. 
# ----------------------------------------------------

SUMMARY_SYSTEM_PROMPT = """
# TODO
"""

# ----------------------------------------------------
# TODO 2: SUMMARY_USER_PROMPT_TEMPLATE 완성
# ----------------------------------------------------
# 요구사항:
# 1. {transcript_snippet}는 전사본이 들어갈 placeholder입니다.
# 2. Instructions 섹션의 내용을 직접 작성해야 합니다.
#    - Overall Summary 요청 
#    - Key Topics 요청 
#    - Topic Structure 요청 (title, summary, image_prompt)
# 3. "Expected JSON Format" 섹션의 JSON Value 부분을 완성하세요.
#    - Key값(overall_summary, title 등)은 그대로 유지
#    - Value값에 모델이 생성해야 할 데이터의 제약조건이나 설명을 작성
# ----------------------------------------------------

SUMMARY_USER_PROMPT_TEMPLATE = """
## Transcript of the Lecture

```
{transcript_snippet}
```

## Instructions
# TODO


## Expected JSON Format

```json
{{
  "overall_summary": "TODO", 
  "topics": [
    {{
      "title": "TODO", // 예: "주제의 핵심을 요약한 짧은 제목"
      "summary": "TODO", // 예: "2~3문장으로 구성된 주제 설명"
      "image_prompt": "TODO" // 예: "토픽에 대한 이미지 생성을 위한 프롬프트"
    }},
    ...
  ]
}}
```

"""

## 4. 요약(Summarization) 모듈

- **역할**
  - 전사된 텍스트를 입력받아 **전체 요약**과 **토픽 리스트**를 생성
  - Response API의 `parse` 기능을 사용하여 구조화된 JSON 형태로 응답 받기

- **구성**
  - `Topic`: 각 토픽의 `title`, `summary`, `image_prompt` 필드를 정의하는 Pydantic 모델
  - `SummaryResponse`: 전체 요약(`overall_summary`)과 토픽 리스트(`topics`)를 포함하는 응답 구조
  - `make_topics_from_transcript`: 전사 텍스트를 입력받아 요약 및 토픽 리스트를 생성하는 함수

- **포인트**
  - `gpt-6.1-sol` 모델과 Response API의 `parse` 기능을 사용하여 구조화된 출력 보장
  - 각 토픽은 이미지 생성을 위한 `image_prompt`를 포함
  - 반환값은 전체 요약 문자열과 토픽 딕셔너리 리스트

In [ ]:
from typing import List
from pydantic import BaseModel

# OpenAI 클라이언트 초기화
make_topics_client = OpenAI()

class Topic(BaseModel):
    title: str
    summary: str
    image_prompt: str

class SummaryResponse(BaseModel):
    overall_summary: str
    topics: List[Topic]


def make_topics_from_transcript(transcript: str):
    """전체 전사 텍스트를 Response API에 넣고, 전체 요약 + 토픽 리스트를 JSON 형태로 받아옵니다."""
    context = [
        {"role": "developer", "content": SUMMARY_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": SUMMARY_USER_PROMPT_TEMPLATE.format(
                transcript_snippet=transcript
            ),
        },
    ]

    res = make_topics_client.responses.parse(
        model="gpt-6.1-sol",
        input=context,
        text_format=SummaryResponse,
    )

    data = res.output_parsed
    return data.overall_summary, [topic.model_dump() for topic in data.topics]

## 5. STT (Speech-to-Text) 모듈

- **역할**
  - YouTube 영상에서 추출한 오디오를 **텍스트 전사본(transcript)** 으로 변환

- **구성**
  - `transcribe_audio_to_text`: 오디오 파일과 STT 모델을 입력받아 텍스트로 전사하는 함수

- **포인트**
  - `gpt-transcribe` 등 STT 모델을 선택해 사용 가능

In [ ]:
# ----------------------------------------------------
# TODO 3: transcribe_audio_to_text 구현
# ----------------------------------------------------
# 요구사항:
# 1. 입력으로 받은 오디오 파일을 STT 모델로 전사하는 함수입니다.
# 2. 아래 조건을 반영해서 구현하세요:
#    - response_format은 반드시 "text" 여야 합니다.
#    - language는 영어 전사를 위해 "en"으로 설정하세요.
#    - model은 자유롭게 지정해도 됩니다.
#    - temperature는 자유롭게 지정해도 됩니다. (예: 0, 0.1, 0.5 등)
# 3. 출력은 문자열(전사된 텍스트)이어야 합니다.
# ----------------------------------------------------

stt_client = OpenAI()

def transcribe_audio_to_text(audio_file, model=""):
    """
    - audio_file: 전사할 오디오 파일 경로
    - model: 사용할 STT 모델 이름
    """
    # TODO
    # 힌트: 03-TTS_STT-Lecture.ipynb
    
    pass

## 6. 이미지 생성 모듈

- **역할**
  - 각 요약 항목의 `image_prompt`를 사용하여 이미지를 자동 생성
  - Response API의 `image_generation` 도구를 사용

- **구성**
  - `generate_image`: 단일 이미지 프롬프트를 입력받아 이미지를 생성하고 저장하는 함수
  - `generate_topic_images`: 여러 토픽의 `image_prompt`를 순차적으로 처리하여 이미지 리스트를 생성하는 함수

- **포인트**
  - `Style` 문장을 추가하여 일관된 교육용 일러스트 스타일을 유지
  - 각 토픽별로 `topic_1.png`, `topic_2.png` 형태로 저장
  - 생성된 이미지는 PDF 강의 요약본에서 표시

In [ ]:
import base64
from pathlib import Path

# OpenAI 클라이언트 초기화
img_client = OpenAI()

# ----------------------------------------------------
# TODO 4: 이미지 생성 함수 구현하기
# ----------------------------------------------------
# 요구사항:
# 1. image_prompt를 입력받아 이미지 생성 API를 호출해야 합니다.
# 2. Response API의 이미지 생성 도구(image_generation)를 사용하세요.
# 3. 스타일 가이드를 prompt에 포함시키도록 구성하세요.
# 4. 반환값은 생성된 이미지 파일 경로입니다.
# 
# 아래 코드 블록은 유지되어야 합니다. 
#   image_bytes = base64.b64decode(image_data[0])
#   파일 저장 부분 전체
# ----------------------------------------------------


def generate_image(prompt, out_dir = "results/images", idx = 0):
    """
    - prompt: 이미지 생성용 image_prompt
    - out_dir: 이미지 저장 경로
    - idx: 이미지 파일 번호

    반환:
      이미지 파일의 저장 경로 (문자열)
    """

    # TODO: 스타일 가이드 작성 및 full_prompt 구성
    # 예시:
    # style_guide = "교육용 일러스트, 고해상도"
    # full_prompt = f"{prompt}. Style: {style_guide}"

    # TODO: Response API 호출
    # 힌트: 01-Image Generation-Lecture.ipynb

    # TODO: 응답에서 base64 이미지 데이터 추출
    # 힌트: 01-Image Generation-Lecture.ipynb
    # image_data = [...]
    image_data = [] 


    image_bytes = base64.b64decode(image_data[0])

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"topic_{idx + 1}.png"
    with open(out_path, "wb") as f:
        f.write(image_bytes)
    return str(out_path)


def generate_topic_images(topics, out_dir: str = "results/images"):
    """여러 토픽(topic)에 대한 이미지를 순차적으로 생성합니다."""
    image_paths = []
    for i, topic in enumerate(topics):
        img_path = generate_image(topic["image_prompt"], out_dir=out_dir, idx=i)
        image_paths.append(img_path)
    return image_paths


## 7. PDF 요약본 생성 모듈 

- **역할**
  - 전체 요약 + 토픽별 텍스트 + 이미지 경로를 받아서 PDF 요약본 생성

- **구성**
  - `build_pdf`: 전체 요약(`overall_summary`), 토픽 리스트(`topics`), 이미지 경로 리스트(`image_paths`)를 입력받아 PDF 파일을 생성하는 함수

- **포인트**
  - PDF 파일로 변환

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_LEFT, TA_JUSTIFY

def build_pdf(overall_summary, topics, image_paths, out_path: str = "results/lecture_summary.pdf"):
    """전체 요약, 토픽들, 이미지 경로들을 받아서 PDF를 생성합니다."""
    
    # 한글 폰트 3가지 등록
    fonts = register_korean_fonts()
    
    # PDF 문서 생성
    doc = SimpleDocTemplate(out_path, pagesize=A4,
                            rightMargin=72, leftMargin=72,
                            topMargin=72, bottomMargin=18)
    
    # 스타일 정의
    styles = getSampleStyleSheet()
    
    # 대 제목: ExtraBold
    title_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontName=fonts['extrabold'],
        fontSize=20,
        textColor='#1a1a1a',
        spaceAfter=30,
        alignment=TA_LEFT
    )
    
    # 중 제목 (Overall Summary, Topic 제목): Bold
    heading_style = ParagraphStyle(
        'CustomHeading',
        parent=styles['Heading2'],
        fontName=fonts['bold'],
        fontSize=16,
        textColor='#2c3e50',
        spaceAfter=12,
        spaceBefore=20,
        alignment=TA_LEFT
    )
    
    topic_title_style = ParagraphStyle(
        'TopicTitle',
        parent=styles['Heading3'],
        fontName=fonts['bold'],
        fontSize=14,
        textColor='#34495e',
        spaceAfter=8,
        spaceBefore=16,
        alignment=TA_LEFT
    )
    
    # 내용: Regular
    body_style = ParagraphStyle(
        'BodyText',
        parent=styles['BodyText'],
        fontName=fonts['regular'],
        fontSize=11,
        textColor='#333333',
        spaceAfter=12,
        alignment=TA_JUSTIFY,
        leading=16
    )
    
    # PDF 내용 구성
    story = []
    
    story.append(Paragraph("LLM Lecture Summary (3Blue1Brown)", title_style))
    story.append(Spacer(1, 0.3*inch))
    
    story.append(Paragraph("Overall Summary", heading_style))
    story.append(Paragraph(overall_summary, body_style))
    story.append(Spacer(1, 0.4*inch))
    
    for i, (topic, img_path) in enumerate(zip(topics, image_paths), 1):
        story.append(Paragraph(f"Topic {i}: {topic['title']}", topic_title_style))
        
        if Path(img_path).exists():
            img = Image(img_path, width=4*inch, height=3*inch)
            story.append(img)
            story.append(Spacer(1, 0.2*inch))
        else:
            story.append(Paragraph(f"[Image not found: {img_path}]", body_style))
        
        story.append(Paragraph(topic['summary'], body_style))
        
        if i < len(topics):
            story.append(Spacer(1, 0.3*inch))
    
    doc.build(story)
    print(f"PDF saved to: {out_path}")
    return out_path

## 8. 전체 파이프라인 정리

- **역할**
  - STT → 강의 요약 → 이미지 생성 → PDF 요약본 생성 end-to-end 파이프라인 
- **구성**
  - `run_full_pipeline` : 전체 파이프라인을 실행하는 함수
- **출력**
  - 전사 텍스트
  - 전체 요약
  - 토픽 리스트
  - 이미지 파일 경로
  - PDF 파일 경로
  - 전사 텍스트 저장 경로


In [ ]:

from pathlib import Path


def run_full_pipeline(audio_file: str):
    """전체 파이프라인을 실행합니다.

    Args:
        audio_file: WAV/WEBM 등 오디오 파일 경로

    Returns:
        dict: transcript_text, overall_summary, topics, image_paths, pdf_path, transcript_path
    """
    
    
    # ----------------------------------------------------
    # TODO 5: 오디오 파일 → 텍스트 전사
    #   - 입력으로 받은 오디오 파일을 STT 모델을 사용해 텍스트로 변환하세요.
    #   - 전사된 텍스트를 transcript_text 변수에 저장하세요.
    # ----------------------------------------------------

    transcript_text = None  # TODO: 실제 코드로 변경하세요.
    print("Transcript length:", len(transcript_text))


    # 1-1) 전사 결과를 파일로 저장 (transcribe 폴더)
    transcribe_dir = Path("results/transcribe")
    audio_name = Path(audio_file).stem 
    transcript_path = transcribe_dir / f"{audio_name}.txt"
    transcript_path.write_text(transcript_text, encoding="utf-8")
    print(f"Transcript saved to: {transcript_path}")

    # ----------------------------------------------------
    # TODO 6: 전사 텍스트 → 요약 + 토픽 리스트 생성
    #   - 전사된 텍스트를 기반으로 전체 요약과 토픽 리스트를 생성하세요.
    #   - 반환된 전체 요약을 overall_summary, 토픽 리스트를 topics 변수에 저장하세요.
    # ----------------------------------------------------

    overall_summary = None  # TODO
    topics = []            # TODO
    print("Number of topics:", len(topics))

    # ----------------------------------------------------
    # TODO 7: 토픽 기반 이미지 생성
    #   - 각 토픽의 image_prompt를 활용해 이미지를 생성하세요.
    #   - 생성된 이미지 파일 경로들을 image_paths 리스트에 저장하세요.
    # ----------------------------------------------------
    
    image_paths = []  # TODO

    # ----------------------------------------------------
    # TODO 8: PDF 생성
    #   - 요약(overall_summary), 토픽(topics), 생성된 이미지(image_paths)를 활용하여
    #     PDF 파일을 만드는 함수를 호출하세요.
    #   - 완성된 PDF의 저장 경로를 pdf_path 변수에 저장하세요.
    # ----------------------------------------------------

    pdf_path = None  # TODO
    print("Summary saved to:", pdf_path)

    # ----------------------------------------------------
    # 결과 반환 
    # ----------------------------------------------------

    return {
        "transcript_text": transcript_text,
        "overall_summary": overall_summary,
        "topics": topics,
        "image_paths": image_paths,
        "pdf_path": pdf_path,
        "transcript_path": str(transcript_path),
    }


## 9. 파이프라인 실행

- **입력 파일 예시**
  - `data/audio/3b1b_llm_lecture.webm`

- **실행 방법**
  - 아래 셀에서 `audio_path`를 원하는 파일로 수정 후 실행

- **결과**
  - 전사 텍스트 파일 (`results/transcribe/`)
  - 이미지 파일들 (`results/images/`)
  - PDF 요약본 (`results/lecture_summary.pdf`)


In [ ]:
audio_path = "data/audio/3b1b_llm_lecture.webm" 

# 전체 파이프라인 실행
results = run_full_pipeline(audio_path)

# 주요 결과 확인
print("\n=== Overall Summary ===")
print(results["overall_summary"])

print("\n=== Number of Topics ===")
print(len(results["topics"]))

print("\n=== PDF Path ===")
print(results["pdf_path"])
